In [ ]:
# 必要なものを用意
from transformers import pipeline
distilled_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)

In [ ]:
# モデルのアテンション重みをプロット
import matplotlib.pyplot as plt

state_dict = pipe.model.state_dict()
weights = state_dict["distilbert.transformer.layer.0.attention.out_lin.weight"]
plt.hist(weights.flatten().cpu().numpy(), bins=250, range=(-0.3, 0.3), edgecolor="C0")
plt.show()

In [ ]:
# スケール係数計算
zero_point = 0
scale = (weights.max() - weights.min()) / (127 - (-128))

In [ ]:
# 量子化テンソルを計算
(weights / scale + zero_point).clamp(-128, 127).round().char()

In [ ]:
# torch.quantize_per_tensor を用いる方法
import torch
from torch import quantize_per_tensor

dtype = torch.qint8
quantized_weights = quantize_per_tensor(weights, scale=scale, zero_point=zero_point, dtype=dtype)
quantized_weights.int_repr()

In [ ]:
## 量子化テンソルを戻してプロット
dequantized_weights = quantized_weights.dequantize()
plt.hist(dequantized_weights.flatten().cpu().numpy(), bins=250, range=(-0.3, 0.3), edgecolor="C0")
plt.show()

In [ ]:
%%timeit # マジックコマンドはセルの先頭
weights @ weights

In [ ]:
# 量子化テンソルの計算準備
from torch.nn .quantized import QFunctional

q_fn = QFunctional()

In [ ]:
# 量子化演算は CPU バックエンドで試す
quantized_weights_cpu = quantized_weights.cpu()

In [ ]:
%%timeit
q_fn.mul(quantized_weights_cpu, quantized_weights_cpu)

In [ ]:
# テンソルのサイズ比較
import sys

sys.getsizeof(weights.storage()) / sys.getsizeof(quantized_weights_cpu.storage())

In [ ]:
# 動的量子化
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.quantization import quantize_dynamic

model_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu"))

model_qunantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)